In [4]:
import pandas as pd
import numpy as np

NUM_COLS = [f'I{i}' for i in range(1, 14)]
CAT_COLS = [f'C{i}' for i in range(1, 27)]
ALL_COLS = ['label'] + NUM_COLS + CAT_COLS

df = pd.read_csv('kaggle-dataset/train.txt', sep='\t', header=None, names=ALL_COLS, nrows=100000)

print(df.shape)
print(df['label'].value_counts())
df.head()

(100000, 40)
label
0    77337
1    22663
Name: count, dtype: int64


,label,I1,I2,I3,I4,I5,I6,I7,I8,I9,...,C17,C18,C19,C20,C21,C22,C23,C24,C25,C26
0,0,1.0,1,5.0,0.0,1382.0,4.0,15.0,2.0,181.0,...,e5ba7672,f54016b9,21ddcdc9,b1252a9d,07b5194c,NaN,3a171ecb,c5c50484,e8b83407,9727dd16
1,0,2.0,0,44.0,1.0,102.0,8.0,2.0,2.0,4.0,...,07c540c4,b04e4670,21ddcdc9,5840adea,60f6221e,NaN,3a171ecb,43f13e8b,e8b83407,731c3655
2,0,2.0,0,1.0,14.0,767.0,89.0,4.0,2.0,245.0,...,8efede7f,3412118d,NaN,NaN,e587c466,ad3062eb,3a171ecb,3b183c5c,NaN,NaN
3,0,NaN,893,NaN,NaN,4392.0,NaN,0.0,0.0,0.0,...,1e88c74f,74ef3502,NaN,NaN,6b3a5ca6,NaN,3a171ecb,9117a34a,NaN,NaN
4,0,3.0,-1,NaN,0.0,2.0,0.0,3.0,0.0,0.0,...,1e88c74f,26b3c7a7,NaN,NaN,21c9516a,NaN,32c7478e,b34f3128,NaN,NaN


In [6]:
from collections import Counter

counter = {col: Counter() for col in CAT_COLS}
total = 0

for chunk in pd.read_csv('kaggle-dataset/train.txt', sep='\t', header=None,
                          names=ALL_COLS, chunksize=1_000_000):
    for col in CAT_COLS:
        counter[col].update(chunk[col].fillna('unknown').astype(str).values)
    total += len(chunk)
    print(f'{total/1e6:.0f}M 처리중...', end='\r')

print('완료')

KeyboardInterrupt: 

In [ ]:
counter

NameError: name 'counter' is not defined

In [ ]:
MIN_FREQ = 10

vocab = {}
for col in CAT_COLS:
    vocab[col] = {'unknown': 0}  # 0: unknown 고정
    idx = 1
    for token, cnt in counter[col].items():
        if token == 'unknown':  # unknown은 스킵
            continue
        if cnt >= MIN_FREQ:
            vocab[col][token] = idx
            idx += 1
            
print('min_freq 적용 후 vocab 크기:')
for col in CAT_COLS:
    print(f'{col}: {len(vocab[col])}')

min_freq 적용 후 vocab 크기:
C1: 1458
C2: 555
C3: 193948
C4: 138800
C5: 306
C6: 18
C7: 11970
C8: 634
C9: 4
C10: 42646
C11: 5178
C12: 192772
C13: 3175
C14: 27
C15: 11422
C16: 181074
C17: 11
C18: 4654
C19: 2031
C20: 4
C21: 189656
C22: 17
C23: 16
C24: 59696
C25: 85
C26: 45570


In [21]:
import pickle
import os

os.makedirs('data/processed', exist_ok=True)

with open('data/processed/vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

print('vocab 저장 완료')

vocab 저장 완료


In [7]:
import pickle

with open('data/processed/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

print('vocab 로드 완료')

vocab 로드 완료


In [9]:
vocab

{'C1': {'unknown': 0,
  '68fd1e64': 1,
  '287e684f': 2,
  '8cf07265': 3,
  '05db9164': 4,
  '439a44a4': 5,
  '241546e0': 6,
  'be589b51': 7,
  '5a9ed9b0': 8,
  '3c9d8785': 9,
  '1464facd': 10,
  '7e5c2ff4': 11,
  '9a89b36c': 12,
  'fb174e6b': 13,
  '5bfa8ab5': 14,
  '87552397': 15,
  'ae82ea21': 16,
  'f473b8dc': 17,
  '1a5f926e': 18,
  '6e4a368f': 19,
  'b455c6d7': 20,
  '17f69355': 21,
  '68a25dc5': 22,
  '09ca0b81': 23,
  '9684fd4d': 24,
  'ff004ae3': 25,
  '75ac2fe6': 26,
  '5ebc3192': 27,
  '39af2607': 28,
  '45cb84c9': 29,
  '36a5b3ff': 30,
  'a86f8721': 31,
  '4a4e85c4': 32,
  'c512b859': 33,
  '41edac3d': 34,
  'cf1f182b': 35,
  'f434fac1': 36,
  'f0a33555': 37,
  '8a033483': 38,
  '8c6ba407': 39,
  '3b65d647': 40,
  '24eda356': 41,
  '65aada8c': 42,
  '9ac48ebf': 43,
  '43b5ce4b': 44,
  '9660b97b': 45,
  '6ca3af46': 46,
  '80e4d755': 47,
  '561bf9d4': 48,
  'd9baf3a8': 49,
  'd4b08d58': 50,
  'fbc55dae': 51,
  'dbe63c2b': 52,
  '64e77ae7': 53,
  '2b92c0d2': 54,
  'ba454362': 5

In [10]:
import pandas as pd
import numpy as np

TRAIN_SIZE = 40_000_000
CHUNKSIZE = 1_000_000

train_nums, train_cats, train_labels = [], [], []
val_nums, val_cats, val_labels = [], [], []
row_count = 0

for chunk in pd.read_csv('kaggle-dataset/train.txt', sep='\t', header=None,
                          names=ALL_COLS, chunksize=CHUNKSIZE):
    
    # numerical 전처리
    num = chunk[NUM_COLS].fillna(0).clip(lower=0)
    num = np.log1p(num).astype(np.float32)
    
    # categorical 전처리
    cat = np.zeros((len(chunk), len(CAT_COLS)), dtype=np.int32)
    for i, col in enumerate(CAT_COLS):
        cat[:, i] = chunk[col].fillna('unknown').astype(str).map(
            lambda x, v=vocab[col]: v.get(x, 0)  # 없는 값은 0(unknown)
        ).values
    
    label = chunk['label'].values.astype(np.int8)
    
    end = row_count + len(chunk)
    
    if row_count < TRAIN_SIZE:
        train_end = min(len(chunk), TRAIN_SIZE - row_count)
        train_nums.append(num.values[:train_end])
        train_cats.append(cat[:train_end])
        train_labels.append(label[:train_end])
    
    if end > TRAIN_SIZE:
        val_start = max(0, TRAIN_SIZE - row_count)
        val_nums.append(num.values[val_start:])
        val_cats.append(cat[val_start:])
        val_labels.append(label[val_start:])
    
    row_count = end
    print(f'{row_count/1e6:.0f}M 처리중...', end='\r')

print('\n저장 중...')
np.save('data/processed/train_num.npy', np.concatenate(train_nums))
np.save('data/processed/train_cat.npy', np.concatenate(train_cats))
np.save('data/processed/train_label.npy', np.concatenate(train_labels))
np.save('data/processed/val_num.npy', np.concatenate(val_nums))
np.save('data/processed/val_cat.npy', np.concatenate(val_cats))
np.save('data/processed/val_label.npy', np.concatenate(val_labels))
print('완료!')

46M 처리중...
저장 중...
완료!


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

class CriteoDataset(Dataset):
    def __init__(self, data_dir, split):
        self.num = np.load(f'{data_dir}/{split}_num.npy', mmap_mode='r')
        self.cat = np.load(f'{data_dir}/{split}_cat.npy', mmap_mode='r')
        self.label = np.load(f'{data_dir}/{split}_label.npy', mmap_mode='r')
    
    def __len__(self):
        return len(self.label)
    
    def __getitem__(self, idx):
        return {
            'num': torch.tensor(self.num[idx], dtype=torch.float32),
            'cat': torch.tensor(self.cat[idx], dtype=torch.long),
            'label': torch.tensor(self.label[idx], dtype=torch.float32)
        }

# 테스트
train_ds = CriteoDataset('data/processed', 'train')
print(f'train: {len(train_ds):,}')

val_ds = CriteoDataset('data/processed', 'val')
print(f'val: {len(val_ds):,}')

train: 40,000,000
val: 5,840,617


In [13]:
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True, 
                          num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=4096, shuffle=False, 
                        num_workers=4, pin_memory=True)

print(f'train 배치 수: {len(train_loader):,}')
print(f'val 배치 수: {len(val_loader):,}')

train 배치 수: 9,766
val 배치 수: 1,426


In [14]:
import torch.nn as nn

class LogisticRegression(nn.Module):
    def __init__(self, num_dim, field_dims):
        super().__init__()
        self.num_linear = nn.Linear(num_dim, 1, bias=False)
        self.embeddings = nn.ModuleList([
            nn.Embedding(fd, 1, padding_idx=0) for fd in field_dims
        ])
        self.bias = nn.Parameter(torch.zeros(1))
    
    def forward(self, num, cat):
        num_part = self.num_linear(num)
        cat_part = sum(emb(cat[:, i]) for i, emb in enumerate(self.embeddings))
        return (num_part + cat_part + self.bias).squeeze(-1)

# field_dims 만들기
field_dims = [len(vocab[col]) for col in CAT_COLS]
print(f'field_dims: {field_dims[:5]}...')
print(f'총 파라미터 예상: {sum(field_dims):,}')

model = LogisticRegression(num_dim=13, field_dims=field_dims)
print(f'모델 파라미터: {sum(p.numel() for p in model.parameters()):,}')

field_dims: [1458, 555, 193948, 138800, 306]...
총 파라미터 예상: 1,085,727
모델 파라미터: 1,085,741


In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    n = 0
    for i, batch in enumerate(loader):
        num = batch['num'].to(device)
        cat = batch['cat'].to(device)
        label = batch['label'].to(device)
        
        optimizer.zero_grad()
        logit = model(num, cat)
        loss = criterion(logit, label)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(label)
        n += len(label)
        
        if i % 50 == 0:  # 500배치마다 출력
            print(f'batch {i}/{len(loader)} | loss: {total_loss/n:.4f}')
    
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        num = batch['num'].to(device)
        cat = batch['cat'].to(device)
        logit = model(num, cat).cpu()
        all_logits.append(logit)
        all_labels.append(batch['label'])
    
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1 / (1 + np.exp(-logits))
    
    return {
        'auc': roc_auc_score(labels, probs),
        'logloss': log_loss(labels, probs)
    }

# 1 epoch 학습
train_loss = train_epoch(model, train_loader, optimizer)
val_metrics = evaluate(model, val_loader)
print(f'train loss: {train_loss:.4f}')
print(f'val AUC: {val_metrics["auc"]:.4f}')
print(f'val logloss: {val_metrics["logloss"]:.4f}')

device: cuda
